# WRN-28-10 RAMP fine-tune baseline (official code, no AAT)

Question: is the weak-class gap (cat / deer / dog / bird) a PreActResNet-18 artifact?
Method `ramp_wrn` in `scripts/ramp_official.sh` runs the official command (`RAMP_cifar10_aug.py`, RobustBench `Gowal2020Uncovering_28_10_extra`, lr 0.01, 3 epochs, `--kl --max`, no GP) and
`aat.evaluate` then runs the same fixed-eps AutoAttack protocol as the PreAct runs (config `official_eval_wrn.yaml`).

**Needs:** `HF_TOKEN` secret, GPU T4 x2, Internet on, and the Carmon 500K aux file `ti_500K_pseudo_labeled.pickle` in `data/` or on the Hub under `cache/` (the official script loads it). Memory is tight on a T4: set `WRN_BS` (e.g. 64) if it runs out of memory.
The first epoch's `[epoch] 1 [time] ... s` log line is the real timing.

In [ ]:
import os
os.environ['HF_REPO'] = 'matokebryan/aat-checkpoints'
os.environ['NTFY_TOPIC'] = 'aat-matoke-7q4r9x'   # subscribe to this in the ntfy app
METHOD = 'ramp_wrn'
RUN = 'ramp_wrn_ft'
# os.environ['WRN_BS'] = '64'   # uncomment if the official batch size (128) does not fit
os.environ['TIME_BUDGET_H'] = '10.5'             # unused by ramp_wrn (3 epochs, no resume)
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
%cd /kaggle/working
!git clone -q -b matt/funny-planck-3j7md5 https://github.com/mattobryan/AAT.git 2>/dev/null || (cd AAT && git pull -q)
%cd /kaggle/working/AAT

def stage(name, cmd):
    """Run a stage, stream its output, push a phone notification, and stop the notebook if it fails."""
    get_ipython().system(f'python scripts/stage.py {name} "{cmd}"')
    if _exit_code != 0:
        raise SystemExit(f'stage {name} failed (exit {_exit_code}): see the notification or logs/{name}.txt on Hugging Face')

stage('install', 'bash scripts/ramp_official.sh install')   # dependencies + full import check, before anything else
stage('data', 'bash scripts/ramp_official.sh data')         # CIFAR-10 (cached on Hugging Face after the first run)

## Official WRN-28-10 fine-tune, seeds 0 and 1 in parallel, then evaluation and comparison with the PreAct baselines

In [ ]:
def finished(run):
    return os.path.exists(f'external/ramp/trained_models/{run}/log_eval_final.txt')

stage('wrn_data', 'bash scripts/ramp_official.sh wrn_data')   # aux pickle: local, else HF cache/, else a clear error
stage('ramp_wrn_train', 'bash scripts/ramp_official.sh train2 ramp_wrn 0 1')
if all(finished(f'{RUN}_s{s}') for s in (0, 1)):
    stage('ramp_wrn_eval', 'bash scripts/ramp_official.sh eval2 ramp_wrn 0 1 && '
          'python scripts/compare_runs.py --names WRN RAMP_PreAct '
          '--a \\"runs_official/ramp_wrn_ft_s*/eval_autoattack.json\\" '
          '--b \\"baselines/results/ramp_scratch_l5_s*_eval_autoattack.json\\"')
else:
    print('Training not finished; check the logs.')